# Colab AI Chat Connector

## Save your login once (recommended)

In Colab, open the **🔑 Secrets** panel and add these two secrets:

- `AIWB_EMAIL` — the email you use for Colab AI Chat
- `AIWB_PASSWORD` — your Colab AI Chat password

Allow this notebook to access both secrets. After that, future sessions require **no email/password typing**.

Then tap ▶ on the next cell. If either secret is missing, the connector will safely prompt only for the missing value.

The notebook never prints your password and contains no private key. Google Colab Secrets stay outside the notebook file.

In [ ]:
import json, time, getpass, urllib.request, urllib.error, urllib.parse
from datetime import datetime, timezone

try:
    from google.colab import ai, userdata
except Exception as exc:
    raise RuntimeError("Google Colab AI is not available in this runtime. Make sure this notebook is open in Google Colab, then reconnect and run again.") from exc

SUPABASE_URL = "https://udyqxlnexnxhmslsebre.supabase.co"
SUPABASE_KEY = "sb_publishable_IgyY0SZrN1tZUPNRnHWVrw_OOZwEsrH"

print("Colab AI Chat connector")

def saved_secret(name):
    try:
        value = userdata.get(name)
        return value.strip() if isinstance(value, str) and value.strip() else None
    except Exception:
        return None

email = saved_secret("AIWB_EMAIL")
password = saved_secret("AIWB_PASSWORD")

if email and password:
    print("Using saved Colab Secrets ✓")
else:
    print("Saved login not fully configured.")
    print("For zero-typing next time, add AIWB_EMAIL and AIWB_PASSWORD in Colab's 🔑 Secrets panel.")
    if not email:
        email = input("App email: ").strip()
    if not password:
        password = getpass.getpass("App password: ")

access_token = None
refresh_token = None
token_started = 0.0

def request(method, path, data=None, authenticated=True, prefer=None):
    headers = {"apikey": SUPABASE_KEY, "Content-Type": "application/json"}
    if authenticated and access_token:
        headers["Authorization"] = "Bearer " + access_token
    if prefer:
        headers["Prefer"] = prefer
    body = None if data is None else json.dumps(data).encode("utf-8")
    req = urllib.request.Request(SUPABASE_URL + path, data=body, headers=headers, method=method)
    try:
        with urllib.request.urlopen(req, timeout=60) as resp:
            raw = resp.read().decode("utf-8")
            return json.loads(raw) if raw else None
    except urllib.error.HTTPError as exc:
        detail = exc.read().decode("utf-8", errors="replace")
        raise RuntimeError(f"Supabase HTTP {exc.code}: {detail}") from None

def sign_in():
    global access_token, refresh_token, token_started
    session = request(
        "POST", "/auth/v1/token?grant_type=password",
        {"email": email, "password": password}, authenticated=False
    )
    access_token = session["access_token"]
    refresh_token = session["refresh_token"]
    token_started = time.monotonic()
    return session["user"]["id"]

def refresh_session():
    global access_token, refresh_token, token_started
    session = request(
        "POST", "/auth/v1/token?grant_type=refresh_token",
        {"refresh_token": refresh_token}, authenticated=False
    )
    access_token = session["access_token"]
    refresh_token = session["refresh_token"]
    token_started = time.monotonic()

user_id = sign_in()
del password
started_at = datetime.now(timezone.utc).isoformat()

def maybe_refresh():
    if time.monotonic() - token_started > 45 * 60:
        refresh_session()

def worker_upsert(status, include_started=False):
    maybe_refresh()
    now = datetime.now(timezone.utc).isoformat()
    payload = {
        "user_id": user_id,
        "last_seen_at": now,
        "updated_at": now,
        "status": status,
        "model": "google-colab-ai",
        "device": "Google Colab AI",
    }
    if include_started:
        payload["started_at"] = started_at
    request(
        "POST", "/rest/v1/aiwb_workers?on_conflict=user_id",
        payload, prefer="resolution=merge-duplicates,return=minimal"
    )

def get_next_job():
    maybe_refresh()
    query = urllib.parse.urlencode({
        "select": "id,prompt",
        "user_id": "eq." + user_id,
        "status": "eq.queued",
        "order": "created_at.asc",
        "limit": "1",
    })
    return request("GET", "/rest/v1/aiwb_jobs?" + query) or []

def patch_job(job_id, values, require_queued=False):
    maybe_refresh()
    params = {"id": "eq." + job_id, "user_id": "eq." + user_id}
    if require_queued:
        params["status"] = "eq.queued"
    query = urllib.parse.urlencode(params)
    return request(
        "PATCH", "/rest/v1/aiwb_jobs?" + query, values,
        prefer="return=representation"
    ) or []

print("Testing Google Colab AI...")
try:
    test = ai.generate_text("Reply with exactly: COLAB_AI_OK")
    test_text = getattr(test, "text", None) or str(test)
    if "COLAB_AI_OK" not in test_text:
        print("AI responded, but with unexpected text:", test_text[:120])
except Exception as exc:
    raise RuntimeError("Colab AI test failed: " + str(exc)) from exc

worker_upsert("ready", include_started=True)
print("\nCONNECTED ✓")
print("Return to https://colab-ai-workbench.vercel.app")
print("Keep this cell running while you chat. It stops after 30 idle minutes.\n")

last_job = time.monotonic()
try:
    while True:
        worker_upsert("ready")
        jobs = get_next_job()
        if jobs:
            job = jobs[0]
            claimed = patch_job(job["id"], {"status": "running", "error": None}, require_queued=True)
            if claimed:
                try:
                    worker_upsert("busy")
                    response = ai.generate_text(job["prompt"])
                    answer = getattr(response, "text", None) or str(response)
                    patch_job(job["id"], {
                        "status": "completed",
                        "result": answer[:100000],
                        "error": None,
                    })
                except Exception as exc:
                    patch_job(job["id"], {
                        "status": "failed",
                        "error": str(exc)[:2000],
                    })
                finally:
                    last_job = time.monotonic()
                    worker_upsert("ready")
        else:
            if time.monotonic() - last_job > 1800:
                print("Idle 30 minutes — connector stopped. Run this cell again when needed.")
                break
            time.sleep(1)
finally:
    try:
        worker_upsert("stopped")
    except Exception:
        pass
